In [ ]:
# --- Step 1: 设置环境 ---

# Fork 项目
!git clone https://github.com/Cl0udTide/happy-llm
%cd happy-llm

# 安装所有依赖
!git checkout my-experiments
%cd ./my_experiments/_3_training_pipeline
%pip install -r ./requirements.txt

In [ ]:
# ======================================================
# Step 2: 数据准备
# ======================================================
from datasets import load_dataset
import os

# 创建数据目录
os.makedirs("./data/wikipedia_cn", exist_ok=True)
os.makedirs("./data/alpaca_gpt4_zh", exist_ok=True)

# 下载并保存 Wikipedia 数据
print("正在下载 Wikipedia-CN 数据集...")
full_wiki_dataset = load_dataset("pleisto/wikipedia-cn-20230720-filtered", split="train")
print(f"完整维基百科数据集大小: {len(full_wiki_dataset)} 条")

subset_size = len(full_wiki_dataset)
print(f"将使用部分数据: {subset_size} 条")
# 使用 .select() 方法来创建一个子集
subset_wiki_dataset = full_wiki_dataset.select(range(subset_size))

subset_wiki_dataset.to_json("./data/wikipedia_cn/pretrain_data_subset.jsonl")
print("Wikipedia-CN 的子集保存成功！")


# 下载并保存 Alpaca 数据
print("\n正在下载 Alpaca-GPT4-ZH 数据集...")
alpaca_dataset = load_dataset("c-s-ale/alpaca-gpt4-data-zh", split="train")
print(f"完整 Alpaca-GPT4-ZH 数据集大小: {len(alpaca_dataset)} 条")

subset_size = len(alpaca_dataset)
print(f"将使用部分数据: {subset_size} 条")
# 使用 .select() 方法来创建一个子集
subset_alpaca_dataset = alpaca_dataset.select(range(subset_size))

subset_alpaca_dataset.to_json("./data/alpaca_gpt4_zh/sft_data.jsonl")
print("Alpaca-GPT4-ZH 的子集保存成功！")

In [ ]:
# ======================================================
# Step 3: 启动预训练
# ======================================================

!python pretrain.py \
    --out_dir "./outputs/tiny_llama_pretrained_wiki" \
    --data_path "./data/wikipedia_cn/pretrain_data_subset.jsonl" \
    --use_swanlab \
    --epochs 1 \
    --batch_size 32 \
    --accumulation_steps 8 \
    --learning_rate 3e-4 \
    --dtype "float16" \
    --gpus "0"

In [ ]:
# ======================================================
# Step 4: 启动监督微调 (SFT)
# ======================================================

!python finetune.py \
    --base_model_path "./outputs/tiny_llama_pretrained_wiki/pretrain_256_4_8192.pth" \
    --out_dir "./outputs/tiny_llama_sft_alpaca" \
    --data_path "./data/alpaca_gpt4_zh/sft_data.jsonl" \
    --use_swanlab \
    --epochs 1 \
    --batch_size 32 \
    --accumulation_steps 8 \
    --learning_rate 2e-5 \
    --dtype "float16" \
    --gpus "0"

In [ ]:
# ======================================================
# Step 5: 评估对比 Pretrain vs. SFT
# ======================================================

# --- 测试 Pretrain 模型 ---
print("--- 正在测试 Pretrain 模型 ---")
!python generate.py \
    --model_path "./outputs/tiny_llama_pretrained_wiki/" \
    --prompt "中国的首都是哪里？"

# --- 测试 SFT 模型 ---
print("\n--- 正在测试 SFT 模型 ---")
# 使用 Chat Template 提问
sft_prompt = "### Instruction:\n中国的首都是哪里？\n\n### Response:\n"
!python generate.py \
    --model_path "./outputs/tiny_llama_sft_alpaca/" \
    --prompt "{sft_prompt}"